# Testing New Records (20-Question Survey)
Loading the saved Decision Tree model, entering normal 20-question survey responses, automatically calculating composite scores, and predicting mental health risk.

In [1]:
import pandas as pd
import joblib

### Load the saved Decision Tree model

In [2]:
model = joblib.load('final_decision_tree_model.pkl')
print("Model loaded successfully")

Model loaded successfully


### Helper function to calculate composite scores from 20 survey answers

In [3]:
def evaluate_patient(survey):
    # 1. Sum up the 13 mood & stress symptoms (1 to 10)
    symptoms = [
        survey['Work_Stress_Level'], survey['Financial_Stress'], survey['Feeling_Sad_Down'],
        survey['Anxious_Nervous'], survey['Loneliness'], survey['Sleep_Trouble'],
        survey['Fatigue'], survey['Feeling_Worthless'], survey['Irritability'],
        survey['Concentration_Difficulty'], survey['Poor_Appetite_Or_Overeating'],
        survey['Compulsive_Behavior'], survey['Obsessive_Thoughts']
    ]
    
    # 2. Daily Distress = symptoms minus protective factors
    total_distress = sum(symptoms) - survey['Social_Support'] - survey['Job_Satisfaction']
    
    # 3. Severe Clinical Flags (0 or 1)
    severe_risk = (
        survey['Suicidal_Thoughts'] * 3 +
        survey['Self_Harm_Thoughts'] * 3 +
        survey['Family_History_Mental_Illness'] * 2 +
        survey['Trauma_History'] * 2 +
        survey['On_Medication']
    )
    
    # 4. Overall Combined Risk Index
    overall_risk = total_distress + (severe_risk * 5)
    
    # Return dataframe ready for model prediction
    return pd.DataFrame([{
        'Overall_Risk_Index': overall_risk,
        'Total_Distress_Score': total_distress,
        'Severe_Risk_Score': severe_risk,
        'Social_Support': survey['Social_Support']
    }])

### Define sample patients with 20 real survey answers

In [4]:
# Person A: Low stress, good sleep, supportive family, no clinical flags
person_a_answers = {
    'Work_Stress_Level': 3,
    'Financial_Stress': 3,
    'Feeling_Sad_Down': 2,
    'Anxious_Nervous': 3,
    'Loneliness': 2,
    'Sleep_Trouble': 3,
    'Fatigue': 3,
    'Feeling_Worthless': 1,
    'Irritability': 2,
    'Concentration_Difficulty': 2,
    'Poor_Appetite_Or_Overeating': 2,
    'Compulsive_Behavior': 1,
    'Obsessive_Thoughts': 1,
    'Social_Support': 8,          # Strong social circle
    'Job_Satisfaction': 8,
    'Family_History_Mental_Illness': 0,
    'Trauma_History': 0,
    'On_Medication': 0,
    'Self_Harm_Thoughts': 0,
    'Suicidal_Thoughts': 0
}

# Person B: High burnout, insomnia, isolation, positive family history
person_b_answers = {
    'Work_Stress_Level': 8,
    'Financial_Stress': 8,
    'Feeling_Sad_Down': 7,
    'Anxious_Nervous': 8,
    'Loneliness': 7,
    'Sleep_Trouble': 9,
    'Fatigue': 8,
    'Feeling_Worthless': 6,
    'Irritability': 7,
    'Concentration_Difficulty': 7,
    'Poor_Appetite_Or_Overeating': 6,
    'Compulsive_Behavior': 5,
    'Obsessive_Thoughts': 6,
    'Social_Support': 2,          # Very isolated
    'Job_Satisfaction': 3,
    'Family_History_Mental_Illness': 1, # Has genetic risk
    'Trauma_History': 1,
    'On_Medication': 0,
    'Self_Harm_Thoughts': 0,
    'Suicidal_Thoughts': 0
}

### Make predictions on Person A and Person B

In [5]:
for name, answers in [('Person A', person_a_answers), ('Person B', person_b_answers)]:
    features_df = evaluate_patient(answers)
    pred = model.predict(features_df)[0]
    
    risk_idx = features_df['Overall_Risk_Index'].values[0]
    distress = features_df['Total_Distress_Score'].values[0]
    status = 'Healthy (Class 0)' if pred == 0 else 'At Risk (Class 1)'
    
    print(f"{name}:")
    print(f"  Total Distress Score: {distress}")
    print(f"  Overall Risk Index  : {risk_idx} (Cutoff is 54.50)")
    print(f"  Model Prediction    : {status}\n")

Person A:
  Total Distress Score: 12
  Overall Risk Index  : 12 (Cutoff is 54.50)
  Model Prediction    : Healthy (Class 0)

Person B:
  Total Distress Score: 87
  Overall Risk Index  : 107 (Cutoff is 54.50)
  Model Prediction    : At Risk (Class 1)

